# Driving: a CPU-only image-feature reference experiment

**Start in Colab with Python 3.12 and a CPU runtime**

1. Open https://colab.research.google.com/ on a computer, choose
   **Upload notebook**, and upload this `.ipynb`. Keep the CPU runtime.
2. Run the first five code cells in order: setup, restore check,
   dependencies, data preparation, and six-view review. The setup
   cell contains embedded source code and needs no manual changes.
3. Stop at the review interface and label the images using the rubric.
   Run cells individually: displaying images does not pause **Run all**.
   The training checks reject incomplete labels.
4. Before pausing, run **Save partial review** and download the backup.
   Colab storage may be lost after a reset; use this same notebook
   to restore the backup in a later session.

Startup checks Python 3.12. Keep the complete error message if a
version check or download fails. The project source is embedded;
a separate GitHub checkout is unnecessary.

The workflow prepares nuScenes mini, saves human six-view review,
extracts frozen ResNet18 features and compares logistic regression,
a small MLP, and matched SAEs with and without reward on CPU.

The representation concatenates six ordered 512-dimensional image
features into 3072 dimensions. It is a separate image-only reference
protocol, separate from Cosmos hidden states and VLM results.
The official pretrained checkpoint is downloaded and hash-verified;
no random weights are substituted. This notebook starts a fresh run;
it does not automatically reproduce the repository's saved results.

Use a runtime whose normal network policy permits official data and
PyTorch model downloads. Review the data terms before acquisition.
Mini's frozen 6/2/2 scene split is a custom experiment allocation;
mini has no official benchmark test split.

Keep WORK on retained storage. Images, reviewed labels and feature
artifacts are not embedded in this notebook. The portable workflow
has synthetic fixture checks; real-image result records are separate.


In [ ]:
#@title Set up the project (leave embedded code unchanged)
import base64, hashlib, io, json, os, shlex, subprocess, sys, zipfile
from pathlib import Path, PurePosixPath

SOURCE_PAYLOAD = '...'
SOURCE_SHA256 = 'adb5f2960526d05f56403785140993ece48656253426b64cc639ae55bcf5751d'
WORK = Path(os.environ.get("DRIVING_WORK_DIR",
    "/content/driving-cpu-work" if Path("/content").is_dir() else "driving-cpu-work")).resolve()
ROOT = WORK / "source"
WORK.mkdir(parents=True, exist_ok=True)
raw_source = base64.b64decode(SOURCE_PAYLOAD)
assert hashlib.sha256(raw_source).hexdigest() == SOURCE_SHA256
with zipfile.ZipFile(io.BytesIO(raw_source)) as bundle:
    source_manifest = json.loads(bundle.read("SOURCE_MANIFEST.json"))
    for name, expected in source_manifest.items():
        relative = PurePosixPath(name)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError("Unsafe source path")
        data = bundle.read(name)
        if hashlib.sha256(data).hexdigest() != expected:
            raise ValueError("Embedded source hash mismatch: " + name)
        path = ROOT / name
        if path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise ValueError("Existing source differs; preserve it and choose a fresh WORK")
        path.parent.mkdir(parents=True, exist_ok=True)
        if not path.exists():
            path.write_bytes(data)
sys.path.insert(0, str(ROOT / "scripts"))
DATA = Path(os.environ.get("DRIVING_DATA_ROOT", str(WORK / "dataset/nuscenes")))
PREPARED, ANNOTATIONS, EXTRACT = WORK/"prepared", WORK/"annotations", WORK/"extract"
FEATURE_PROTOCOL = "cpu-resnet18"
WEIGHTS = WORK / "models/resnet18/resnet18-f37072fd.pth"

def run(args):
    args = [str(arg) for arg in args]
    print(shlex.join(args), flush=True)
    subprocess.run(args, cwd=ROOT, check=True)

print("Workspace:", WORK)
print("Verified source files:", len(source_manifest))


## Storage and optional review restore

Colab's `/content` storage is ephemeral: switching runtime can remove it.
Use retained storage via DRIVING_WORK_DIR before bootstrap, or download
the small review backup cell below before switching. Keep this same
notebook to restore that backup; its source fingerprint is checked.
Images and weights are not included in the review backup. The acquisition
cell can reacquire data while preserving the restored split and decisions.


In [ ]:
#@title Restore a review backup (skip on the first run)
# After uploading a saved review_handoff.zip to a new runtime, set its path here.
REVIEW_BACKUP_TO_RESTORE = None
if REVIEW_BACKUP_TO_RESTORE is not None:
    from portable_handoff import restore_review
    restored = restore_review(Path(REVIEW_BACKUP_TO_RESTORE), WORK, SOURCE_SHA256)
    print("Restored review files:", len(restored["files_sha256"]))


## 1. Fix the proof-of-concept settings before seeing results

Keep these settings for all four models. A changed protocol should use
a new WORK folder. Do not choose another split after seeing test results.


In [ ]:
#@title Install CPU dependencies and freeze settings
import importlib.metadata
SEED, REVIEW_K, EPOCHS, HIDDEN_DIM, TOP_K = 42, 10, 10, 256, 32
LAYER = 1
protocol = dict(seed=SEED, review_k=REVIEW_K, epochs=EPOCHS,
                hidden_dim=HIDDEN_DIM, k=TOP_K, layer=LAYER,
                feature_protocol=FEATURE_PROTOCOL, pooling="camera_concat")
protocol_path = WORK / "requested_protocol.json"
if protocol_path.exists() and json.loads(protocol_path.read_text()) != protocol:
    raise ValueError("Protocol changed; choose a fresh WORK before continuing")
protocol_path.write_text(json.dumps(protocol, indent=2) + "\n")
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("The verified CPU lock requires a Python 3.12 runtime")

def installed_versions():
    return sorted({f"{d.metadata['Name']}=={d.version}"
                   for d in importlib.metadata.distributions() if d.metadata['Name']})

versions_path = WORK / "cpu_package_versions.txt"
if versions_path.exists():
    if versions_path.read_text() != "\n".join(installed_versions()) + "\n":
        raise RuntimeError("Restore recorded CPU packages before resuming this cache")
else:
    run([sys.executable, "-m", "pip", "install", "-r",
         ROOT/"requirements-cpu-lock.txt", "ipywidgets"])
    versions_path.write_text("\n".join(installed_versions()) + "\n")


## 2. Acquire mini, or use an already extracted local copy

The [official tutorial][mini-tutorial]
publishes the mini archive. Its Motional CDN is listed in the
[AWS registry][aws-registry].
Follow the [data terms](https://www.nuscenes.org/terms-of-use).

The downloader records received-byte SHA256, checks transfer completeness
and stages metadata/keyframe camera extraction. It omits sweeps and other
sensors. A local hash is not an independently verified official checksum.
Do not overwrite an existing data folder; preparation validates its files.

[mini-tutorial]: https://github.com/nutonomy/nuscenes-devkit/blob/master/python-sdk/tutorials/nuscenes_tutorial.ipynb
[aws-registry]: https://github.com/awslabs/open-data-registry/blob/main/datasets/motional-nuscenes.yaml


In [ ]:
#@title Download mini and prepare six-camera views
required_tables = ["scene", "sample", "sample_data", "calibrated_sensor", "sensor"]
if all((DATA / "v1.0-mini" / (name + ".json")).is_file()
       for name in required_tables) and (DATA / "samples").is_dir():
    print("Using local data; preparation will validate references and selected images.")
else:
    run([sys.executable, ROOT/"scripts/acquire_nuscenes_mini.py",
         "--archive", WORK/"downloads/v1.0-mini.tgz", "--data-root", DATA,
         "--report", WORK/"acquisition.json"])
if not PREPARED.exists():
    run([sys.executable, ROOT/"scripts/prepare_driving.py", "prepare",
         "--data-root", DATA, "--version", "v1.0-mini", "--frames-per-scene", 8,
         "--seed", SEED, "--rubric-file", ROOT/"latest_grading_criteria.md",
         "--output-dir", PREPARED])
print(json.loads((PREPARED/"prepare_metadata.json").read_text()))


## 3. Review six views on CPU

Enter your reviewer identifier. Each explicit label click saves the CSV
atomically; rerunning this cell reloads it. Use the frozen rubric and keep
uncertain cases uncertain. A model score is not a human label. Preserve
independent review records and adjudicate before importing one final CSV.


In [ ]:
#@title Open manual review (pause here to label images)
from notebook_review import ReviewSession
review = ReviewSession(PREPARED, WORK/"reviewed.csv")
print(review.progress())
review.widget(DATA)


### Save partial review before pausing

Review can span multiple sessions. This cell exports saved
decisions, frozen splits and settings, including incomplete
review. Images and models are excluded. In a later session,
open this same notebook, upload the backup and set its path
in the restore cell above.


In [ ]:
#@title Save partial review and download a small backup
if not (WORK/"reviewed.csv").is_file():
    print("Save a review decision before running this cell.")
else:
    from portable_handoff import pack_review
    partial_backup = WORK/"partial_review_handoff.zip"
    pack_review(WORK, partial_backup, SOURCE_SHA256)
    try:
        from google.colab import files as colab_files
    except ImportError:
        print("Save this file using your file browser:", partial_backup)
    else:
        colab_files.download(str(partial_backup))


## 4. Freeze the reviewed labels

Pending/uncertain decisions stay excluded. Each split needs both classes
for this experiment contract. If it does not, keep this PoC as evidence of
the limitation and plan more data; never fabricate labels or re-split by
the test result. Use a new annotations directory for revised labels.


In [ ]:
if not ANNOTATIONS.exists():
    run([sys.executable, ROOT/"scripts/prepare_driving.py", "import-labels",
         "--prepared-dir", PREPARED, "--reviews", WORK/"reviewed.csv",
         "--output-dir", ANNOTATIONS])
label_report = json.loads((ANNOTATIONS/"import_metadata.json").read_text())
current_csv_hash = hashlib.sha256((WORK/"reviewed.csv").read_bytes()).hexdigest()
if list(label_report["review_inputs_sha256"].values()) != [current_csv_hash]:
    raise ValueError("Reviews changed after import; preserve annotations and re-import")
print(label_report["support"])
if not label_report["ready_for_current_training_contract"]:
    raise ValueError("Insufficient reviewed class support; extraction/training blocked")


## Save review progress before changing runtimes

This creates a small backup of prepared records, review CSV, annotations
and the chosen protocol. On Colab it offers a browser download; in other
Jupyter environments retain the printed ZIP using your file browser.
Re-run this cell after recording model/version information to include it.
Preserve feature caches and authorized images separately after extraction.


In [ ]:
from portable_handoff import pack_review
backup_path = WORK / "review_handoff.zip"
pack_review(WORK, backup_path, SOURCE_SHA256)
print("Review backup:", backup_path, "bytes:", backup_path.stat().st_size)
try:
    from google.colab import files as colab_files
except ImportError:
    print("Retain this ZIP before replacing the runtime or workspace.")
else:
    colab_files.download(str(backup_path))


## 5. Download the pinned CPU encoder

This step downloads the official ImageNet1K ResNet18 checkpoint
(about 47 MB), verifies its full SHA256, and preserves existing files.
It installs no additional model library. Keep the recorded CPU
dependencies and preprocessing fixed while completing this cache.


In [ ]:
frozen_review = json.loads((ANNOTATIONS/"import_metadata.json").read_text())
reviewed_hash = hashlib.sha256((WORK/"reviewed.csv").read_bytes()).hexdigest()
if (not frozen_review["ready_for_current_training_contract"]
        or list(frozen_review["review_inputs_sha256"].values()) != [reviewed_hash]):
    raise RuntimeError("Complete and import the reviewed labels before extraction")
if versions_path.read_text() != "\n".join(installed_versions()) + "\n":
    raise RuntimeError("CPU environment changed since preparation")
run([sys.executable, ROOT/"scripts/acquire_cpu_encoder.py", "--output", WEIGHTS])


## 6. Measure one sample, then complete the same CPU cache

One sample contains six views. The frozen encoder performs real CPU
inference with the verified pretrained weights. Use one writer and
storage supporting atomic rename/file sync. Preserve the first smoke
report, then complete all selected samples before training.

Layer 1 and the legacy `*_mean.npy` name are storage compatibility
aliases for this protocol's camera concatenation; they do not claim
decoder-layer MLP or token-mean features.


In [ ]:
import shutil

def extract_command(limit):
    return [sys.executable, ROOT/"scripts/extract_cpu.py",
            "--prepared-dir", PREPARED, "--data-root", DATA,
            "--weights", WEIGHTS, "--threads", 2, "--save-every", 1,
            "--max-samples", limit, "--output-dir", EXTRACT]

run(extract_command(1))
smoke_report = WORK/"smoke_extraction_summary.json"
if not smoke_report.exists():
    shutil.copy2(EXTRACT/"extraction_summary.json", smoke_report)
print(json.loads(smoke_report.read_text()))


In [ ]:
if versions_path.read_text() != "\n".join(installed_versions()) + "\n":
    raise RuntimeError("CPU environment changed since smoke")
run(extract_command(0))
summary = json.loads((EXTRACT/"extraction_summary.json").read_text())
if not summary.get("selection_complete"):
    raise RuntimeError("Extraction is partial; training remains blocked")
run([sys.executable, ROOT/"scripts/run_real_driving.py", "validate",
     "--prepared-dir", PREPARED, "--annotations-dir", ANNOTATIONS,
     "--extract-dir", EXTRACT, "--layer", LAYER, "--pooling", "mean",
     "--feature-protocol", FEATURE_PROTOCOL,
     "--output", WORK/"handoff_validation.json"])


## 7. Run the controlled CPU comparison

This launches Logistic regression, a small MLP and matched SAE with and
without tail reward. It reuses the imported manifest and checks saved
evidence before training; it never re-splits scenes. Start with one seed.
Preserve exact commands, per-sample predictions and failures. A mini-only
table remains a custom PoC even if the scores look good.


In [ ]:
if versions_path.read_text() != "\n".join(installed_versions()) + "\n":
    raise RuntimeError("CPU environment changed since extraction")
RUNS = WORK / "comparison_runs"
run([sys.executable, ROOT/"scripts/run_real_driving.py", "train",
     "--prepared-dir", PREPARED, "--annotations-dir", ANNOTATIONS,
     "--extract-dir", EXTRACT, "--output-dir", RUNS, "--layer", LAYER,
     "--pooling", "mean", "--feature-protocol", FEATURE_PROTOCOL,
     "--review-k", REVIEW_K, "--seed", SEED, "--epochs", EPOCHS,
     "--hidden-dim", HIDDEN_DIM, "--k", TOP_K])
print((RUNS/"comparison/main_table.csv").read_text())


## 8. Inspect and retain results

For the browser, run the command below in a terminal that can reach its
localhost server. A notebook output cell does not provide hosted access
to that server. Keep its trained activation cache and authorized images.
Include real FP/FN cases in the final demonstration; random weights and
synthetic examples are not model-effectiveness evidence.

Copy WORK using a transfer method permitted by your environment. Include
hidden `.checkpoints/`, model revision, package versions, frozen prepared
records, review CSV, annotations, extraction and comparison outputs.
The source bundle contains no real data or weights and may be shared
separately. Respect data/model redistribution conditions.


In [ ]:
browser_command = [sys.executable, ROOT/"scripts/neuron_explorer.py", "serve",
                   "--run", RUNS/"sae_abstopk_norm", "--samples-root", DATA/"samples"]
print(shlex.join([str(arg) for arg in browser_command]))
print("Keep the complete work directory:", WORK)
